# 🎬 YouTube Viral Clipper - Heatmap + FFmpeg (30s)

Este notebook:
1. **Monitora** canais YouTube por views e viralização
2. **Histórico** JSON para evitar duplicação
3. **Download** com yt-dlp (robusto contra bloqueios)
4. **Heatmap via análise de áudio** para detectar picos de ação
5. **Corte de 30 segundos** (15s antes + 15s depois do pico)
6. **Conversão FFmpeg** para 9:16 vertical (TikTok/Instagram/Facebook)

In [ ]:
# Instalação das dependências
!pip install -q yt-dlp pandas tqdm numpy google-api-python-client

In [ ]:
# CONFIGURAÇÕES
YOUTUBE_API_KEY = "SUA_API_KEY_AQUI"  # <-- COLOQUE SUA API KEY

CHANNEL_IDS = [
    "UCX6OQ3DkcsbYNE6H8uQQuVA",  # Exemplo
]

MIN_VIEWS = 50000
MAX_RESULTS_PER_CHANNEL = 5
CLIP_DURATION = 30  # 30 segundos total
OUTPUT_DIR = "/content/viral_clips"
HISTORY_FILE = "/content/video_history.json"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Configurações carregadas. Clips de {CLIP_DURATION}s")

In [ ]:
# SISTEMA DE HISTÓRICO
import json
from datetime import datetime

def load_history():
    if os.path.exists(HISTORY_FILE):
        with open(HISTORY_FILE, 'r') as f:
            return json.load(f)
    return []

def save_to_history(video_id, info):
    history = load_history()
    history.append({
        "video_id": video_id,
        "processed_at": datetime.now().isoformat(),
        "title": info.get("title", ""),
        "views": info.get("views", 0)
    })
    with open(HISTORY_FILE, 'w') as f:
        json.dump(history, f, indent=2)

def is_processed(video_id):
    return any(r["video_id"] == video_id for r in load_history())

print("✅ Sistema de histórico pronto")

In [ ]:
# MONITORAMENTO DE CANAIS
from googleapiclient.discovery import build

def get_viral_videos(channel_ids):
    youtube = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)
    videos = []
    
    for ch_id in channel_ids:
        try:
            search = youtube.search().list(
                part='snippet',
                channelId=ch_id,
                order='viewCount',
                type='video',
                maxResults=MAX_RESULTS_PER_CHANNEL
            ).execute()
            
            ids = [i['id']['videoId'] for i in search['items']]
            details = youtube.videos().list(
                part='snippet,statistics,contentDetails',
                id=','.join(ids)
            ).execute()
            
            for v in details['items']:
                stats = v['statistics']
                views = int(stats.get('viewCount', 0))
                likes = int(stats.get('likeCount', 0))
                
                if views >= MIN_VIEWS:
                    videos.append({
                        'video_id': v['id'],
                        'title': v['snippet']['title'],
                        'views': views,
                        'likes': likes,
                        'engagement': (likes / views) if views > 0 else 0,
                        'duration': v['contentDetails']['duration']
                    })
        except Exception as e:
            print(f"Erro canal {ch_id}: {e}")
    
    videos.sort(key=lambda x: x['engagement'], reverse=True)
    return videos

print("✅ Monitoramento pronto")

In [ ]:
# DOWNLOAD COM YT-DLP
import yt_dlp

def download_video(video_id):
    url = f"https://www.youtube.com/watch?v={video_id}"
    
    ydl_opts = {
        'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
        'outtmpl': os.path.join(OUTPUT_DIR, f'{video_id}.%(ext)s'),
        'merge_output_format': 'mp4',
        'quiet': False,
        'no_warnings': False,
    }
    
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        filename = ydl.prepare_filename(info)
        # Merge se necessário
        if not os.path.exists(filename):
            filename = filename.replace('.webm', '.mp4').replace('.mkv', '.mp4')
        
        return filename, info.get('title', video_id)

print("✅ Download com yt-dlp pronto")

In [ ]:
# ANÁLISE DE HEATMAP VIA ÁUDIO (FFPROBE + FFMPEG)
import subprocess
import numpy as np

def analyze_heatmap_ffmpeg(video_path):
    """
    Analisa picos de áudio para simular heatmap de momentos virais.
    Retorna o timestamp do pico mais alto (momento mais provável de ação).
    """
    # Extrai dados de volume ao longo do tempo
    cmd = [
        'ffprobe', '-v', 'error',
        '-f', 'lavfi',
        '-i', f'amovie={video_path},astats=metadata=1:reset=1',
        '-show_entries', 'frame=pkt_pts_time:frame_tags=lavfi.astats.Overall.RMS_level',
        '-of', 'csv=p=0'
    ]
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
        lines = result.stdout.strip().split('\n')
        
        peaks = []
        for line in lines:
            if ',' in line:
                parts = line.split(',')
                if len(parts) >= 2:
                    try:
                        time_sec = float(parts[0])
                        rms = float(parts[1])
                        peaks.append((time_sec, rms))
                    except:
                        pass
        
        if not peaks:
            # Fallback: meio do vídeo
            probe = subprocess.run(
                ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
                 '-of', 'csv=p=0', video_path],
                capture_output=True, text=True
            )
            duration = float(probe.stdout.strip())
            return duration / 2, duration
        
        # Encontra pico máximo (maior RMS = mais barulho/ação)
        max_peak = max(peaks, key=lambda x: x[1])
        
        # Pega duração total
        probe = subprocess.run(
            ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
             '-of', 'csv=p=0', video_path],
            capture_output=True, text=True
        )
        duration = float(probe.stdout.strip())
        
        return max_peak[0], duration
    
    except Exception as e:
        print(f"Erro na análise: {e}")
        # Fallback
        probe = subprocess.run(
            ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
             '-of', 'csv=p=0', video_path],
            capture_output=True, text=True
        )
        duration = float(probe.stdout.strip()) if probe.stdout.strip() else 60
        return duration / 2, duration

print("✅ Análise de heatmap pronta")

In [ ]:
# CORTE E CONVERSÃO COM FFMPEG
import re

def sanitize_filename(title):
    # Remove caracteres inválidos
    title = re.sub(r'[<>:"/\\|?*]', '', title)
    title = re.sub(r'\s+', '_', title)
    return title[:50]  # Limita tamanho

def create_vertical_clip(video_path, start_time, duration=CLIP_DURATION, output_dir=OUTPUT_DIR):
    """
    Corta 30s e converte para 9:16 vertical usando FFmpeg.
    Mantém proporção com crop centralizado.
    """
    base_name = os.path.basename(video_path)
    video_id = base_name.split('.')[0]
    output_filename = f"{video_id}_clip_30s.mp4"
    output_path = os.path.join(output_dir, output_filename)
    
    # Ajusta start_time para garantir que caiba no vídeo
    probe = subprocess.run(
        ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
         '-of', 'csv=p=0', video_path],
        capture_output=True, text=True
    )
    total_duration = float(probe.stdout.strip())
    
    # Garante que o clip de 30s caiba
    if start_time < duration / 2:
        start_time = duration / 2
    if start_time > total_duration - duration / 2:
        start_time = total_duration - duration / 2
    
    actual_start = start_time - (duration / 2)
    
    # Comando FFmpeg: crop central + escala para 1080x1920
    cmd = [
        'ffmpeg', '-y',
        '-ss', str(actual_start),
        '-t', str(duration),
        '-i', video_path,
        '-vf', 'crop=ih*(9/16):ih:(iw-ih*(9/16))/2:0,scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920',
        '-c:a', 'copy',
        '-movflags', '+faststart',
        output_path
    ]
    
    print(f"🎬 Cortando e convertendo... {duration}s a partir de {actual_start:.1f}s")
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode == 0 and os.path.exists(output_path):
        print(f"✅ Clip criado: {output_path}")
        return output_path
    else:
        print(f"❌ Erro FFmpeg: {result.stderr}")
        return None

print("✅ Conversão vertical pronta")

In [ ]:
# PROCESSAMENTO PRINCIPAL
from tqdm import tqdm

def process_all():
    print("🔍 Buscando vídeos virais...")
    videos = get_viral_videos(CHANNEL_IDS)
    print(f"📺 Encontrados {len(videos)} vídeos potenciais\n")
    
    processed_count = 0
    
    for video in tqdm(videos, desc="Processando"):
        vid_id = video['video_id']
        
        if is_processed(vid_id):
            print(f"⏭️  Já processado: {video['title'][:50]}...")
            continue
        
        print(f"\n{'='*60}")
        print(f"🎬 PROCESSANDO: {video['title']}")
        print(f"📊 Views: {video['views']:,} | Engajamento: {video['engagement']:.2%}")
        print(f"{'='*60}")
        
        try:
            # Download
            video_path, title = download_video(vid_id)
            
            # Análise de heatmap
            peak_time, total_duration = analyze_heatmap_ffmpeg(video_path)
            print(f"🔥 Momento viral detectado: {peak_time:.1f}s (duração total: {total_duration:.1f}s)")
            
            # Corte e conversão
            clip_path = create_vertical_clip(video_path, peak_time, CLIP_DURATION)
            
            if clip_path:
                save_to_history(vid_id, {
                    "title": title,
                    "views": video['views'],
                    "clip_path": clip_path,
                    "viral_moment": peak_time
                })
                processed_count += 1
            
            # Limpeza opcional (comentar se quiser manter originais)
            # os.remove(video_path)
            
        except Exception as e:
            print(f"❌ Erro ao processar vídeo: {e}")
            import traceback
            traceback.print_exc()
    
    print(f"\n✅ Processamento concluído! {processed_count} clips criados.")
    print(f"📁 Clips salvos em: {OUTPUT_DIR}")

# Executar
if YOUTUBE_API_KEY == "SUA_API_KEY_AQUI":
    print("⚠️  Configure sua API Key do YouTube primeiro!")
else:
    process_all()